In [3]:
# ============================================================
# QCTrojan-Bench: Quantum Circuit Trojan Detection Benchmark
# ============================================================
# Notebook:  S2_tampered_circuits.ipynb
# Purpose:   Generate 6,000 tampered circuits (static +
#            triggered) derived from S1 benign circuits
# Authors:   Zeeshan Ajmal
#            University of Oulu, Finland
# Version:   QCTrojan-Bench v1.0
# License:   CC BY 4.0
# ============================================================
#
# CELL 1 — Imports and Configuration
#
# All paths, seeds, and constants defined here.
# No other cell hardcodes any value.
# Run this cell first.
#
# Prerequisites:
#   S1_benign_circuits.ipynb must be complete and validated.
#   All 3,000 benign circuits must exist in dataset/circuits/benign/
# ============================================================

import os
import json
import random
from pathlib import Path
from datetime import datetime, timezone

from qiskit import QuantumCircuit, qpy
from qiskit.circuit.library import MCPhaseGate

# ── Versioning ───────────────────────────────────────────────
QISKIT_VERSION    = "2.3.1"
GENERATOR_VERSION = "QCTrojan-Bench-v1.0"
DATASET_VERSION   = "v1"

# ── Paths (auto-detected) ─────────────────────────────────────
NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent
DATASET_ROOT = PROJECT_ROOT / "dataset"

BENIGN_DIR   = DATASET_ROOT / "circuits" / "benign"
TAMPERED_DIR = DATASET_ROOT / "circuits" / "tampered"

# ── Dataset constants ─────────────────────────────────────────
NUM_BENIGN   = 600   # benign circuits per family
TROJAN_TYPES = ["static", "triggered"]

# ── Per-family tamper seeds ───────────────────────────────────
# Each family has two seeds: one per Trojan type.
# Kept separate for full reproducibility and auditability.
TAMPER_SEEDS = {
    "deutsch_jozsa": {"static": 11000, "triggered": 12000},
    "grover":        {"static": 21000, "triggered": 22000},
    "qaoa":          {"static": 31000, "triggered": 32000},
    "vqc":           {"static": 41000, "triggered": 42000},
    "qft":           {"static": 51000, "triggered": 52000},
}

FAMILIES = list(TAMPER_SEEDS.keys())

# ── Create output directories ─────────────────────────────────
for family in FAMILIES:
    for trojan_type in TROJAN_TYPES:
        (TAMPERED_DIR / family / DATASET_VERSION /
         trojan_type / "circuits").mkdir(parents=True, exist_ok=True)
        (TAMPERED_DIR / family / DATASET_VERSION /
         trojan_type / "metadata").mkdir(parents=True, exist_ok=True)

# ── Verify S1 output exists ───────────────────────────────────
print("Verifying S1 output...")
missing = []
for family in FAMILIES:
    circuit_dir = BENIGN_DIR / family / DATASET_VERSION / "circuits"
    found = len(list(circuit_dir.glob("*.qpy")))
    if found != NUM_BENIGN:
        missing.append(f"{family}: found {found}, expected {NUM_BENIGN}")

if missing:
    print("ERROR — S1 output incomplete:")
    for m in missing:
        print(f"  {m}")
    raise RuntimeError("Complete S1 before running S2.")

# ── Summary ───────────────────────────────────────────────────
print("=" * 55)
print("QCTrojan-Bench — S2 Tampered Circuit Generation")
print("=" * 55)
print(f"  Generator version  : {GENERATOR_VERSION}")
print(f"  Qiskit version     : {QISKIT_VERSION}")
print(f"  Families           : {FAMILIES}")
print(f"  Trojan types       : {TROJAN_TYPES}")
print(f"  Circuits per type  : {NUM_BENIGN * len(FAMILIES)}")
print(f"  Total expected     : {NUM_BENIGN * len(FAMILIES) * len(TROJAN_TYPES):,}")
print(f"  Output directory   : {TAMPERED_DIR}")
print("=" * 55)
print("Cell 1 complete. Directories created. Ready for Cell 2.")


Verifying S1 output...
QCTrojan-Bench — S2 Tampered Circuit Generation
  Generator version  : QCTrojan-Bench-v1.0
  Qiskit version     : 2.3.1
  Families           : ['deutsch_jozsa', 'grover', 'qaoa', 'vqc', 'qft']
  Trojan types       : ['static', 'triggered']
  Circuits per type  : 3000
  Total expected     : 6,000
  Output directory   : C:\Users\zajmal23\OneDrive - University of Oulu and Oamk\quantum_circuits_exp\dataset\circuits\tampered
Cell 1 complete. Directories created. Ready for Cell 2.


In [4]:
# ============================================================
# CELL 2 — Shared Trojan Builder Functions
# ============================================================
#
# Defines the two Trojan construction strategies used across
# all five algorithm families.
#
# Type A — Static Trojan
#   CX–RZ–CX motif injected into the circuit body.
#   Always active. Corrupts output on every execution.
#   No new qubits added.
#
# Type B — Triggered Trojan
#   MCPhaseGate controlled by one existing data qubit.
#   Payload dormant when control qubit is |0> (normal execution).
#   Activates only if control qubit is |1>.
#   No new qubits added.
#
# Both types:
#   - Preserve circuit width (qubit count unchanged)
#   - Preserve syntactic validity
#   - Corrupt output probability distribution via phase shifts
#   - Use only existing data qubits
#
# These functions are called by all per-family generation cells.
# Do not modify these functions after generation begins.
# ============================================================

def build_static_trojan_block(
    n_qubits: int,
    rng: random.Random,
) -> tuple:
    """
    Build a static (always-active) Trojan block.

    Construction: CX–RZ–CX motif on two randomly selected
    existing data qubits. The RZ angle is chosen from a fixed
    set of non-trivial values to ensure measurable phase corruption.

    Args:
        n_qubits: number of qubits in the host circuit
        rng:      seeded Random instance for determinism

    Returns:
        (block: QuantumCircuit, meta: dict)
        block operates on n_qubits qubits (same width as host)
    """
    if n_qubits < 2:
        raise ValueError(
            f"Static Trojan requires at least 2 qubits, got {n_qubits}"
        )

    # Select two distinct qubits
    control, target = rng.sample(range(n_qubits), 2)

    # Non-trivial RZ angles that produce measurable phase corruption
    rz_angle = rng.choice([0.25, 0.5, 0.75, 1.0, 1.25])

    block = QuantumCircuit(n_qubits, name="trojan_static")
    block.cx(control, target)
    block.rz(rz_angle, target)
    block.cx(control, target)

    meta = {
        "trojan_type":        "static",
        "mod_class":          "A",
        "control_qubit":      control,
        "target_qubit":       target,
        "rz_angle":           rz_angle,
        "payload_gate_count": 3,
        "payload_description": (
            f"CX-RZ({rz_angle:.2f})-CX on qubits "
            f"[{control}, {target}]"
        ),
    }

    return block, meta


def build_triggered_trojan_block(
    n_qubits: int,
    rng: random.Random,
) -> tuple:
    """
    Build a triggered (controlled-activation) Trojan block.

    Construction: MCPhaseGate controlled by one existing data
    qubit, applied to a different target qubit. The payload is
    dormant when the control qubit is |0> (normal execution
    condition). It would activate only if the control qubit
    were |1>, which does not occur under normal execution.

    No ancilla qubits are added. The control qubit is an
    existing data qubit chosen deterministically per circuit.
    This avoids trivial detection via qubit-count inspection.

    Args:
        n_qubits: number of qubits in the host circuit
        rng:      seeded Random instance for determinism

    Returns:
        (block: QuantumCircuit, meta: dict)
        block operates on n_qubits qubits (same width as host)
    """
    if n_qubits < 2:
        raise ValueError(
            f"Triggered Trojan requires at least 2 qubits, "
            f"got {n_qubits}"
        )

    # Select control and target qubits
    control, target = rng.sample(range(n_qubits), 2)

    # Phase angle — non-trivial to ensure detectable corruption
    # if ever activated
    phase_angle = rng.choice([0.5, 1.0, 1.5, 2.0, 2.5])

    block = QuantumCircuit(n_qubits, name="trojan_triggered")

    # MCPhaseGate(angle, num_ctrl_qubits) applied to [control, target]
    gate = MCPhaseGate(phase_angle, num_ctrl_qubits=1)
    block.append(gate, [control, target])

    meta = {
        "trojan_type":        "triggered",
        "mod_class":          "B",
        "control_qubit":      control,
        "target_qubit":       target,
        "phase_angle":        phase_angle,
        "payload_gate_count": 1,
        "dormancy_condition": "control qubit |0> (normal execution)",
        "payload_description": (
            f"MCPhaseGate({phase_angle:.2f}) controlled by "
            f"qubit {control}, target qubit {target}. "
            f"Dormant when control is |0>."
        ),
    }

    return block, meta


def inject_trojan(
    qc_benign: QuantumCircuit,
    trojan_block: QuantumCircuit,
    insertion_index: int,
) -> QuantumCircuit:
    """
    Inject a Trojan block into a benign circuit at a specific
    instruction index.

    The Trojan block is inserted by splitting the benign circuit
    at insertion_index and composing the three parts:
      prefix + trojan_block + suffix

    Args:
        qc_benign:       original benign circuit
        trojan_block:    Trojan circuit (same qubit width)
        insertion_index: instruction index to insert after

    Returns:
        New QuantumCircuit with Trojan injected
    """
    n = qc_benign.num_qubits

    assert trojan_block.num_qubits == n, (
        f"Trojan block width {trojan_block.num_qubits} does not "
        f"match circuit width {n}"
    )

    instructions = list(qc_benign.data)
    total        = len(instructions)

    # Clamp insertion index to valid range
    insertion_index = max(0, min(insertion_index, total))

    # Build tampered circuit
    qc_tampered = QuantumCircuit(n)

    # Prefix
    for inst in instructions[:insertion_index]:
        qc_tampered.append(inst.operation, inst.qubits, inst.clbits)

    # Trojan block
    qc_tampered.compose(trojan_block, inplace=True)

    # Suffix
    for inst in instructions[insertion_index:]:
        qc_tampered.append(inst.operation, inst.qubits, inst.clbits)

    return qc_tampered


def validate_tampered_circuit(
    qc_tampered: QuantumCircuit,
    qc_benign: QuantumCircuit,
    sample_id: str,
) -> None:
    """
    Validate a tampered circuit against its benign source.

    Checks:
    - Same qubit count (no ancilla added)
    - No measurements
    - More instructions than benign (Trojan was injected)

    Raises AssertionError if any check fails.
    """
    assert qc_tampered.num_qubits == qc_benign.num_qubits, (
        f"{sample_id}: qubit count changed "
        f"({qc_benign.num_qubits} → {qc_tampered.num_qubits})"
    )

    assert qc_tampered.num_clbits == 0, (
        f"{sample_id}: measurement found in tampered circuit"
    )

    assert len(qc_tampered.data) > len(qc_benign.data), (
        f"{sample_id}: tampered circuit not larger than benign "
        f"({len(qc_tampered.data)} <= {len(qc_benign.data)})"
    )


# ── Smoke test ────────────────────────────────────────────────
# Quick sanity check on both builder functions

_test_rng = random.Random(99999)
_test_qc  = QuantumCircuit(4)
_test_qc.h(range(4))
_test_qc.cx(0, 1)
_test_qc.cx(2, 3)

_static_block, _static_meta = build_static_trojan_block(4, _test_rng)
_triggered_block, _triggered_meta = build_triggered_trojan_block(
    4, _test_rng
)

_tampered_static   = inject_trojan(_test_qc, _static_block, 2)
_tampered_triggered = inject_trojan(_test_qc, _triggered_block, 2)

validate_tampered_circuit(_tampered_static,    _test_qc, "smoke_static")
validate_tampered_circuit(_tampered_triggered, _test_qc, "smoke_triggered")

print("Trojan builder smoke test passed.")
print(f"  Static block    : {_static_meta['payload_description']}")
print(f"  Triggered block : {_triggered_meta['payload_description']}")
print(f"  Benign depth    : {_test_qc.depth()}")
print(f"  Static depth    : {_tampered_static.depth()}")
print(f"  Triggered depth : {_tampered_triggered.depth()}")
print("Cell 2 complete. Trojan builders ready for all families.")


Trojan builder smoke test passed.
  Static block    : CX-RZ(1.25)-CX on qubits [0, 1]
  Triggered block : MCPhaseGate(2.00) controlled by qubit 1, target qubit 3. Dormant when control is |0>.
  Benign depth    : 2
  Static depth    : 5
  Triggered depth : 4
Cell 2 complete. Trojan builders ready for all families.


In [5]:
# ============================================================
# CELL 3 — Generate Deutsch-Jozsa Tampered Circuits
# ============================================================
#
# Loads all 600 benign DJ circuits from S1.
# Generates 2 tampered variants per benign circuit:
#   - 600 static Trojans
#   - 600 triggered Trojans
# Total: 1,200 tampered DJ circuits
#
# Insertion point:
#   DJ structure: H-layer → oracle → H-layer
#   Trojans are inserted after the oracle block and before
#   the final H-layer. This is the most realistic insertion
#   point — it mimics a malicious transformation applied
#   after oracle compilation but before interference.
#
# Output:
#   dataset/circuits/tampered/deutsch_jozsa/v1/static/
#   dataset/circuits/tampered/deutsch_jozsa/v1/triggered/
# ============================================================

FAMILY = "deutsch_jozsa"

BENIGN_CIRCUIT_DIR  = BENIGN_DIR / FAMILY / DATASET_VERSION / "circuits"
BENIGN_METADATA_DIR = BENIGN_DIR / FAMILY / DATASET_VERSION / "metadata"

def get_dj_insertion_index(qc: QuantumCircuit) -> int:
    """
    Find insertion index for DJ circuit.

    DJ structure: [H×n] [oracle gates] [H×(n-1)]
    We insert after the oracle, before the final H-layer.

    Strategy: find the last instruction that is NOT an H gate
    on the input qubits. Insert after it.
    If no non-H gate found (constant oracle), insert at midpoint.
    """
    instructions = list(qc.data)
    n = qc.num_qubits

    # Walk backwards to find last non-H instruction
    for i in range(len(instructions) - 1, -1, -1):
        inst = instructions[i]
        if inst.operation.name != "h":
            return i + 1  # insert after this instruction

    # Fallback: constant oracle — insert at midpoint
    return max(1, len(instructions) // 2)


# ── Generation loop ───────────────────────────────────────────

benign_files = sorted(BENIGN_CIRCUIT_DIR.glob("*.qpy"))
assert len(benign_files) == NUM_BENIGN, \
    f"Expected {NUM_BENIGN} benign circuits, found {len(benign_files)}"

stats = {"static": {"generated": 0, "skipped": 0},
         "triggered": {"generated": 0, "skipped": 0}}

for idx, qpy_file in enumerate(benign_files):

    # ── Load benign circuit and metadata ─────────────────────
    with open(qpy_file, "rb") as f:
        qc_benign = qpy.load(f)[0]

    benign_sample_id = qpy_file.stem
    meta_path = BENIGN_METADATA_DIR / f"{benign_sample_id}.json"
    with open(meta_path) as f:
        benign_meta = json.load(f)

    n_qubits         = qc_benign.num_qubits
    insertion_index  = get_dj_insertion_index(qc_benign)

    # ── Generate both Trojan types ────────────────────────────
    for trojan_type in TROJAN_TYPES:

        seed        = TAMPER_SEEDS[FAMILY][trojan_type] + idx
        local_rng   = random.Random(seed)

        sample_id   = (
            f"tampered_{FAMILY}_{trojan_type}_"
            f"{DATASET_VERSION}_{idx:06d}"
        )

        out_circuit_dir  = (TAMPERED_DIR / FAMILY / DATASET_VERSION
                            / trojan_type / "circuits")
        out_metadata_dir = (TAMPERED_DIR / FAMILY / DATASET_VERSION
                            / trojan_type / "metadata")

        qpy_out = out_circuit_dir / f"{sample_id}.qpy"
        if qpy_out.exists():
            stats[trojan_type]["skipped"] += 1
            continue

        # Build Trojan block
        if trojan_type == "static":
            block, trojan_meta = build_static_trojan_block(
                n_qubits, local_rng
            )
        else:
            block, trojan_meta = build_triggered_trojan_block(
                n_qubits, local_rng
            )

        # Inject
        qc_tampered = inject_trojan(
            qc_benign, block, insertion_index
        )

        # Validate
        validate_tampered_circuit(
            qc_tampered, qc_benign, sample_id
        )

        # Save QPY
        with open(qpy_out, "wb") as f:
            qpy.dump(qc_tampered, f)

        # Save metadata
        metadata = {
            "sample_id":          sample_id,
            "label":              "tampered",
            "trojan_type":        trojan_type,
            "trojan_severity":    None,
            "algorithm_family":   FAMILY,
            "algorithm_version":  DATASET_VERSION,
            "parent_sample_id":   benign_sample_id,
            "n_qubits":           n_qubits,
            "insertion_index":    insertion_index,
            "insertion_point":    "after_oracle_before_final_h",
            "oracle_type":        benign_meta.get("oracle_type"),
            "generation_index":   idx,
            "generation_seed":    seed,
            "generator_version":  GENERATOR_VERSION,
            "qiskit_version":     QISKIT_VERSION,
            "created":            datetime.now(timezone.utc).isoformat(),
            **trojan_meta,
        }

        with open(out_metadata_dir / f"{sample_id}.json", "w") as f:
            json.dump(metadata, f, indent=2)

        stats[trojan_type]["generated"] += 1

# ── Validation ────────────────────────────────────────────────

for trojan_type in TROJAN_TYPES:
    out_dir = (TAMPERED_DIR / FAMILY / DATASET_VERSION
               / trojan_type / "circuits")
    files   = sorted(out_dir.glob("*.qpy"))

    assert len(files) == NUM_BENIGN, \
        f"{trojan_type}: expected {NUM_BENIGN}, found {len(files)}"

    for f in files:
        with open(f, "rb") as fh:
            qc = qpy.load(fh)[0]
        assert qc.num_clbits == 0, \
            f"Measurement found in {f.name}"

print(f"Deutsch-Jozsa tampered generation complete.")
for trojan_type in TROJAN_TYPES:
    s = stats[trojan_type]
    print(f"  {trojan_type:10s} — generated: {s['generated']}, "
          f"skipped: {s['skipped']}")
print(f"  Total tampered : "
      f"{sum(s['generated'] for s in stats.values())} circuits")
print(f"  Validation     : all circuits clean ✓")


Deutsch-Jozsa tampered generation complete.
  static     — generated: 0, skipped: 600
  triggered  — generated: 0, skipped: 600
  Total tampered : 0 circuits
  Validation     : all circuits clean ✓


In [6]:
# ============================================================
# CELL 4 — Generate Grover Tampered Circuits
# ============================================================
#
# Loads all 600 benign Grover circuits from S1.
# Generates 2 tampered variants per benign circuit:
#   - 600 static Trojans
#   - 600 triggered Trojans
# Total: 1,200 tampered Grover circuits
#
# Insertion point:
#   Grover structure: H-layer → [oracle → diffusion]×iterations
#   Trojans are inserted between the oracle and diffusion
#   operator in the first iteration. This mimics a malicious
#   transformation applied inside the Grover loop — the most
#   realistic and structurally camouflaged position.
#
# Output:
#   dataset/circuits/tampered/grover/v1/static/
#   dataset/circuits/tampered/grover/v1/triggered/
# ============================================================

FAMILY = "grover"

BENIGN_CIRCUIT_DIR  = BENIGN_DIR / FAMILY / DATASET_VERSION / "circuits"
BENIGN_METADATA_DIR = BENIGN_DIR / FAMILY / DATASET_VERSION / "metadata"


def get_grover_insertion_index(qc: QuantumCircuit) -> int:
    """
    Find insertion index for Grover circuit.

    Grover structure: H-layer, then alternating oracle+diffusion.
    Strategy: skip the initial H-layer, then find the midpoint
    of the remaining instructions. This places the Trojan
    inside the oracle-diffusion block of the first iteration.

    Fallback: insert at 1/3 of total instructions if structure
    cannot be determined.
    """
    instructions = list(qc.data)
    n = qc.num_qubits

    # Find end of initial H-layer (first n instructions are H)
    h_end = 0
    for i, inst in enumerate(instructions):
        if inst.operation.name == "h":
            h_end = i + 1
        else:
            break

    # Insert at midpoint of post-H instructions
    remaining = len(instructions) - h_end
    if remaining > 2:
        return h_end + max(1, remaining // 2)

    # Fallback
    return max(1, len(instructions) // 3)


# ── Generation loop ───────────────────────────────────────────

benign_files = sorted(BENIGN_CIRCUIT_DIR.glob("*.qpy"))
assert len(benign_files) == NUM_BENIGN, \
    f"Expected {NUM_BENIGN} benign circuits, found {len(benign_files)}"

stats = {"static": {"generated": 0, "skipped": 0},
         "triggered": {"generated": 0, "skipped": 0}}

for idx, qpy_file in enumerate(benign_files):

    # ── Load benign circuit and metadata ─────────────────────
    with open(qpy_file, "rb") as f:
        qc_benign = qpy.load(f)[0]

    benign_sample_id = qpy_file.stem
    meta_path = BENIGN_METADATA_DIR / f"{benign_sample_id}.json"
    with open(meta_path) as f:
        benign_meta = json.load(f)

    n_qubits        = qc_benign.num_qubits
    insertion_index = get_grover_insertion_index(qc_benign)

    # ── Generate both Trojan types ────────────────────────────
    for trojan_type in TROJAN_TYPES:

        seed      = TAMPER_SEEDS[FAMILY][trojan_type] + idx
        local_rng = random.Random(seed)

        sample_id = (
            f"tampered_{FAMILY}_{trojan_type}_"
            f"{DATASET_VERSION}_{idx:06d}"
        )

        out_circuit_dir  = (TAMPERED_DIR / FAMILY / DATASET_VERSION
                            / trojan_type / "circuits")
        out_metadata_dir = (TAMPERED_DIR / FAMILY / DATASET_VERSION
                            / trojan_type / "metadata")

        qpy_out = out_circuit_dir / f"{sample_id}.qpy"
        if qpy_out.exists():
            stats[trojan_type]["skipped"] += 1
            continue

        # Build Trojan block
        if trojan_type == "static":
            block, trojan_meta = build_static_trojan_block(
                n_qubits, local_rng
            )
        else:
            block, trojan_meta = build_triggered_trojan_block(
                n_qubits, local_rng
            )

        # Inject
        qc_tampered = inject_trojan(
            qc_benign, block, insertion_index
        )

        # Validate
        validate_tampered_circuit(
            qc_tampered, qc_benign, sample_id
        )

        # Save QPY
        with open(qpy_out, "wb") as f:
            qpy.dump(qc_tampered, f)

        # Save metadata
        metadata = {
            "sample_id":          sample_id,
            "label":              "tampered",
            "trojan_type":        trojan_type,
            "trojan_severity":    None,
            "algorithm_family":   FAMILY,
            "algorithm_version":  DATASET_VERSION,
            "parent_sample_id":   benign_sample_id,
            "n_qubits":           n_qubits,
            "insertion_index":    insertion_index,
            "insertion_point":    "inside_oracle_diffusion_block",
            "oracle_type":        benign_meta.get("oracle_type"),
            "num_iterations":     benign_meta.get("num_iterations"),
            "generation_index":   idx,
            "generation_seed":    seed,
            "generator_version":  GENERATOR_VERSION,
            "qiskit_version":     QISKIT_VERSION,
            "created":            datetime.now(timezone.utc).isoformat(),
            **trojan_meta,
        }

        with open(out_metadata_dir / f"{sample_id}.json", "w") as f:
            json.dump(metadata, f, indent=2)

        stats[trojan_type]["generated"] += 1

# ── Validation ────────────────────────────────────────────────

for trojan_type in TROJAN_TYPES:
    out_dir = (TAMPERED_DIR / FAMILY / DATASET_VERSION
               / trojan_type / "circuits")
    files   = sorted(out_dir.glob("*.qpy"))

    assert len(files) == NUM_BENIGN, \
        f"{trojan_type}: expected {NUM_BENIGN}, found {len(files)}"

    for f in files:
        with open(f, "rb") as fh:
            qc = qpy.load(fh)[0]
        assert qc.num_clbits == 0, \
            f"Measurement found in {f.name}"

print(f"Grover tampered generation complete.")
for trojan_type in TROJAN_TYPES:
    s = stats[trojan_type]
    print(f"  {trojan_type:10s} — generated: {s['generated']}, "
          f"skipped: {s['skipped']}")
print(f"  Total tampered : "
      f"{sum(s['generated'] for s in stats.values())} circuits")
print(f"  Validation     : all circuits clean ✓")


Grover tampered generation complete.
  static     — generated: 0, skipped: 600
  triggered  — generated: 0, skipped: 600
  Total tampered : 0 circuits
  Validation     : all circuits clean ✓


In [8]:
# ============================================================
# CELL 5 — Generate QAOA Tampered Circuits
# ============================================================
#
# Loads all 600 benign QAOA circuits from S1.
# Generates 2 tampered variants per benign circuit:
#   - 600 static Trojans
#   - 600 triggered Trojans
# Total: 1,200 tampered QAOA circuits
#
# Insertion point:
#   QAOA structure: H-layer → [RZZ cost unitary → RX mixer]×p
#   Trojans are inserted inside the cost unitary of the
#   first layer — between the RZZ gates. This is the most
#   realistic position: it mimics a malicious modification
#   of the problem Hamiltonian encoding, which is the most
#   sensitive part of the QAOA circuit.
#
# Note on parameterised circuits:
#   QAOA circuits have symbolic parameters (not bound).
#   Trojan gates use fixed numeric angles — this is correct
#   and intentional. The Trojan is a fixed corruption, not
#   a parameterised one.
#
# Output:
#   dataset/circuits/tampered/qaoa/v1/static/
#   dataset/circuits/tampered/qaoa/v1/triggered/
# ============================================================

FAMILY = "qaoa"

BENIGN_CIRCUIT_DIR  = BENIGN_DIR / FAMILY / DATASET_VERSION / "circuits"
BENIGN_METADATA_DIR = BENIGN_DIR / FAMILY / DATASET_VERSION / "metadata"


def get_qaoa_insertion_index(qc: QuantumCircuit) -> int:
    """
    Find insertion index for QAOA circuit.

    QAOA structure: H-layer, then alternating RZZ and RX gates.
    Strategy: skip the initial H-layer, then insert after the
    first RZZ gate encountered. This places the Trojan inside
    the first cost unitary layer.

    Fallback: insert after the H-layer if no RZZ found.
    """
    instructions = list(qc.data)
    n = qc.num_qubits

    # Find end of initial H-layer
    h_end = 0
    for i, inst in enumerate(instructions):
        if inst.operation.name == "h":
            h_end = i + 1
        else:
            break

    # Find first RZZ after H-layer and insert after it
    for i in range(h_end, len(instructions)):
        if instructions[i].operation.name == "rzz":
            return i + 1

    # Fallback: insert right after H-layer
    return h_end


# ── Generation loop ───────────────────────────────────────────

benign_files = sorted(BENIGN_CIRCUIT_DIR.glob("*.qpy"))
assert len(benign_files) == NUM_BENIGN, \
    f"Expected {NUM_BENIGN} benign circuits, found {len(benign_files)}"

stats = {"static": {"generated": 0, "skipped": 0},
         "triggered": {"generated": 0, "skipped": 0}}

for idx, qpy_file in enumerate(benign_files):

    # ── Load benign circuit and metadata ─────────────────────
    with open(qpy_file, "rb") as f:
        qc_benign = qpy.load(f)[0]

    benign_sample_id = qpy_file.stem
    meta_path = BENIGN_METADATA_DIR / f"{benign_sample_id}.json"
    with open(meta_path) as f:
        benign_meta = json.load(f)

    n_qubits        = qc_benign.num_qubits
    insertion_index = get_qaoa_insertion_index(qc_benign)

    # ── Generate both Trojan types ────────────────────────────
    for trojan_type in TROJAN_TYPES:

        seed      = TAMPER_SEEDS[FAMILY][trojan_type] + idx
        local_rng = random.Random(seed)

        sample_id = (
            f"tampered_{FAMILY}_{trojan_type}_"
            f"{DATASET_VERSION}_{idx:06d}"
        )

        out_circuit_dir  = (TAMPERED_DIR / FAMILY / DATASET_VERSION
                            / trojan_type / "circuits")
        out_metadata_dir = (TAMPERED_DIR / FAMILY / DATASET_VERSION
                            / trojan_type / "metadata")

        qpy_out = out_circuit_dir / f"{sample_id}.qpy"
        if qpy_out.exists():
            stats[trojan_type]["skipped"] += 1
            continue

        # Build Trojan block
        if trojan_type == "static":
            block, trojan_meta = build_static_trojan_block(
                n_qubits, local_rng
            )
        else:
            block, trojan_meta = build_triggered_trojan_block(
                n_qubits, local_rng
            )

        # Inject
        qc_tampered = inject_trojan(
            qc_benign, block, insertion_index
        )

        # Validate
        validate_tampered_circuit(
            qc_tampered, qc_benign, sample_id
        )

        # Save QPY
        with open(qpy_out, "wb") as f:
            qpy.dump(qc_tampered, f)

        # Save metadata
        metadata = {
            "sample_id":          sample_id,
            "label":              "tampered",
            "trojan_type":        trojan_type,
            "trojan_severity":    None,
            "algorithm_family":   FAMILY,
            "algorithm_version":  DATASET_VERSION,
            "parent_sample_id":   benign_sample_id,
            "n_qubits":           n_qubits,
            "insertion_index":    insertion_index,
            "insertion_point":    "inside_cost_unitary_layer_1",
            "p_layers":           benign_meta.get("p_layers"),
            "num_edges":          benign_meta.get("num_edges"),
            "generation_index":   idx,
            "generation_seed":    seed,
            "generator_version":  GENERATOR_VERSION,
            "qiskit_version":     QISKIT_VERSION,
            "created":            datetime.now(timezone.utc).isoformat(),
            **trojan_meta,
        }

        with open(out_metadata_dir / f"{sample_id}.json", "w") as f:
            json.dump(metadata, f, indent=2)

        stats[trojan_type]["generated"] += 1

# ── Validation ────────────────────────────────────────────────

for trojan_type in TROJAN_TYPES:
    out_dir = (TAMPERED_DIR / FAMILY / DATASET_VERSION
               / trojan_type / "circuits")
    files   = sorted(out_dir.glob("*.qpy"))

    assert len(files) == NUM_BENIGN, \
        f"{trojan_type}: expected {NUM_BENIGN}, found {len(files)}"

    for f in files:
        with open(f, "rb") as fh:
            qc = qpy.load(fh)[0]
        assert qc.num_clbits == 0, \
            f"Measurement found in {f.name}"

print(f"QAOA tampered generation complete.")
for trojan_type in TROJAN_TYPES:
    s = stats[trojan_type]
    print(f"  {trojan_type:10s} — generated: {s['generated']}, "
          f"skipped: {s['skipped']}")
print(f"  Total tampered : "
      f"{sum(s['generated'] for s in stats.values())} circuits")
print(f"  Validation     : all circuits clean ✓")


QAOA tampered generation complete.
  static     — generated: 0, skipped: 600
  triggered  — generated: 0, skipped: 600
  Total tampered : 0 circuits
  Validation     : all circuits clean ✓


In [7]:
# ============================================================
# CELL 6 — Generate VQC Tampered Circuits
# ============================================================
#
# Loads all 600 benign VQC circuits from S1.
# Generates 2 tampered variants per benign circuit:
#   - 600 static Trojans
#   - 600 triggered Trojans
# Total: 1,200 tampered VQC circuits
#
# Insertion point:
#   VQC structure: alternating rotation layers and CX
#   entanglement layers repeated for reps cycles.
#   Trojans are inserted after the first CX entanglement
#   layer. This mimics a malicious modification inside the
#   variational ansatz — camouflaged among legitimate
#   entangling operations.
#
# Output:
#   dataset/circuits/tampered/vqc/v1/static/
#   dataset/circuits/tampered/vqc/v1/triggered/
# ============================================================

FAMILY = "vqc"

BENIGN_CIRCUIT_DIR  = BENIGN_DIR / FAMILY / DATASET_VERSION / "circuits"
BENIGN_METADATA_DIR = BENIGN_DIR / FAMILY / DATASET_VERSION / "metadata"


def get_vqc_insertion_index(qc: QuantumCircuit) -> int:
    """
    Find insertion index for VQC circuit.

    VQC structure: rotation gates (RY/RX) followed by CX
    entanglement layers, repeated for reps cycles.
    Strategy: find the first CX gate and insert after it.
    This places the Trojan inside the first entanglement
    layer where it is most camouflaged.

    Fallback: insert at one third of total instructions.
    """
    instructions = list(qc.data)

    # Find first CX gate and insert after it
    for i, inst in enumerate(instructions):
        if inst.operation.name == "cx":
            return i + 1

    # Fallback
    return max(1, len(instructions) // 3)


# ── Generation loop ───────────────────────────────────────────

benign_files = sorted(BENIGN_CIRCUIT_DIR.glob("*.qpy"))
assert len(benign_files) == NUM_BENIGN, \
    f"Expected {NUM_BENIGN} benign circuits, found {len(benign_files)}"

stats = {"static": {"generated": 0, "skipped": 0},
         "triggered": {"generated": 0, "skipped": 0}}

for idx, qpy_file in enumerate(benign_files):

    # ── Load benign circuit and metadata ─────────────────────
    with open(qpy_file, "rb") as f:
        qc_benign = qpy.load(f)[0]

    benign_sample_id = qpy_file.stem
    meta_path = BENIGN_METADATA_DIR / f"{benign_sample_id}.json"
    with open(meta_path) as f:
        benign_meta = json.load(f)

    n_qubits        = qc_benign.num_qubits
    insertion_index = get_vqc_insertion_index(qc_benign)

    # ── Generate both Trojan types ────────────────────────────
    for trojan_type in TROJAN_TYPES:

        seed      = TAMPER_SEEDS[FAMILY][trojan_type] + idx
        local_rng = random.Random(seed)

        sample_id = (
            f"tampered_{FAMILY}_{trojan_type}_"
            f"{DATASET_VERSION}_{idx:06d}"
        )

        out_circuit_dir  = (TAMPERED_DIR / FAMILY / DATASET_VERSION
                            / trojan_type / "circuits")
        out_metadata_dir = (TAMPERED_DIR / FAMILY / DATASET_VERSION
                            / trojan_type / "metadata")

        qpy_out = out_circuit_dir / f"{sample_id}.qpy"
        if qpy_out.exists():
            stats[trojan_type]["skipped"] += 1
            continue

        # Build Trojan block
        if trojan_type == "static":
            block, trojan_meta = build_static_trojan_block(
                n_qubits, local_rng
            )
        else:
            block, trojan_meta = build_triggered_trojan_block(
                n_qubits, local_rng
            )

        # Inject
        qc_tampered = inject_trojan(
            qc_benign, block, insertion_index
        )

        # Validate
        validate_tampered_circuit(
            qc_tampered, qc_benign, sample_id
        )

        # Save QPY
        with open(qpy_out, "wb") as f:
            qpy.dump(qc_tampered, f)

        # Save metadata
        metadata = {
            "sample_id":          sample_id,
            "label":              "tampered",
            "trojan_type":        trojan_type,
            "trojan_severity":    None,
            "algorithm_family":   FAMILY,
            "algorithm_version":  DATASET_VERSION,
            "parent_sample_id":   benign_sample_id,
            "n_qubits":           n_qubits,
            "insertion_index":    insertion_index,
            "insertion_point":    "after_first_cx_entanglement_layer",
            "reps":               benign_meta.get("reps"),
            "rotation_blocks":    benign_meta.get("rotation_blocks"),
            "entanglement":       benign_meta.get("entanglement"),
            "generation_index":   idx,
            "generation_seed":    seed,
            "generator_version":  GENERATOR_VERSION,
            "qiskit_version":     QISKIT_VERSION,
            "created":            datetime.now(timezone.utc).isoformat(),
            **trojan_meta,
        }

        with open(out_metadata_dir / f"{sample_id}.json", "w") as f:
            json.dump(metadata, f, indent=2)

        stats[trojan_type]["generated"] += 1

# ── Validation ────────────────────────────────────────────────

for trojan_type in TROJAN_TYPES:
    out_dir = (TAMPERED_DIR / FAMILY / DATASET_VERSION
               / trojan_type / "circuits")
    files   = sorted(out_dir.glob("*.qpy"))

    assert len(files) == NUM_BENIGN, \
        f"{trojan_type}: expected {NUM_BENIGN}, found {len(files)}"

    for f in files:
        with open(f, "rb") as fh:
            qc = qpy.load(fh)[0]
        assert qc.num_clbits == 0, \
            f"Measurement found in {f.name}"

print(f"VQC tampered generation complete.")
for trojan_type in TROJAN_TYPES:
    s = stats[trojan_type]
    print(f"  {trojan_type:10s} — generated: {s['generated']}, "
          f"skipped: {s['skipped']}")
print(f"  Total tampered : "
      f"{sum(s['generated'] for s in stats.values())} circuits")
print(f"  Validation     : all circuits clean ✓")


VQC tampered generation complete.
  static     — generated: 600, skipped: 0
  triggered  — generated: 600, skipped: 0
  Total tampered : 1200 circuits
  Validation     : all circuits clean ✓


In [8]:
# ============================================================
# CELL 7 — Generate QFT Tampered Circuits
# ============================================================
#
# Loads all 600 benign QFT circuits from S1.
# Generates 2 tampered variants per benign circuit:
#   - 600 static Trojans
#   - 600 triggered Trojans
# Total: 1,200 tampered QFT circuits
#
# Insertion point:
#   QFT structure: alternating H gates and controlled phase
#   rotations (CP gates), followed by optional SWAP network.
#   Trojans are inserted after the first CP gate encountered.
#   This places the Trojan inside the controlled rotation
#   block — camouflaged among the legitimate phase gates
#   which are the structural signature of QFT circuits.
#
# Output:
#   dataset/circuits/tampered/qft/v1/static/
#   dataset/circuits/tampered/qft/v1/triggered/
# ============================================================

FAMILY = "qft"

BENIGN_CIRCUIT_DIR  = BENIGN_DIR / FAMILY / DATASET_VERSION / "circuits"
BENIGN_METADATA_DIR = BENIGN_DIR / FAMILY / DATASET_VERSION / "metadata"


def get_qft_insertion_index(qc: QuantumCircuit) -> int:
    """
    Find insertion index for QFT circuit.

    QFT structure: H gate on first qubit, then controlled
    phase rotations (cp), then H on next qubit, etc.
    Strategy: find the first CP gate and insert after it.
    This places the Trojan inside the rotation block of
    the first qubit's QFT stage.

    Fallback: insert at one quarter of total instructions
    if no CP gate is found.
    """
    instructions = list(qc.data)

    # Find first CP gate and insert after it
    for i, inst in enumerate(instructions):
        if inst.operation.name in ("cp", "p", "cu1"):
            return i + 1

    # Fallback: insert at quarter point
    return max(1, len(instructions) // 4)


# ── Generation loop ───────────────────────────────────────────

benign_files = sorted(BENIGN_CIRCUIT_DIR.glob("*.qpy"))
assert len(benign_files) == NUM_BENIGN, \
    f"Expected {NUM_BENIGN} benign circuits, found {len(benign_files)}"

stats = {"static": {"generated": 0, "skipped": 0},
         "triggered": {"generated": 0, "skipped": 0}}

for idx, qpy_file in enumerate(benign_files):

    # ── Load benign circuit and metadata ─────────────────────
    with open(qpy_file, "rb") as f:
        qc_benign = qpy.load(f)[0]

    benign_sample_id = qpy_file.stem
    meta_path = BENIGN_METADATA_DIR / f"{benign_sample_id}.json"
    with open(meta_path) as f:
        benign_meta = json.load(f)

    n_qubits        = qc_benign.num_qubits
    insertion_index = get_qft_insertion_index(qc_benign)

    # ── Generate both Trojan types ────────────────────────────
    for trojan_type in TROJAN_TYPES:

        seed      = TAMPER_SEEDS[FAMILY][trojan_type] + idx
        local_rng = random.Random(seed)

        sample_id = (
            f"tampered_{FAMILY}_{trojan_type}_"
            f"{DATASET_VERSION}_{idx:06d}"
        )

        out_circuit_dir  = (TAMPERED_DIR / FAMILY / DATASET_VERSION
                            / trojan_type / "circuits")
        out_metadata_dir = (TAMPERED_DIR / FAMILY / DATASET_VERSION
                            / trojan_type / "metadata")

        qpy_out = out_circuit_dir / f"{sample_id}.qpy"
        if qpy_out.exists():
            stats[trojan_type]["skipped"] += 1
            continue

        # Build Trojan block
        if trojan_type == "static":
            block, trojan_meta = build_static_trojan_block(
                n_qubits, local_rng
            )
        else:
            block, trojan_meta = build_triggered_trojan_block(
                n_qubits, local_rng
            )

        # Inject
        qc_tampered = inject_trojan(
            qc_benign, block, insertion_index
        )

        # Validate
        validate_tampered_circuit(
            qc_tampered, qc_benign, sample_id
        )

        # Save QPY
        with open(qpy_out, "wb") as f:
            qpy.dump(qc_tampered, f)

        # Save metadata
        metadata = {
            "sample_id":          sample_id,
            "label":              "tampered",
            "trojan_type":        trojan_type,
            "trojan_severity":    None,
            "algorithm_family":   FAMILY,
            "algorithm_version":  DATASET_VERSION,
            "parent_sample_id":   benign_sample_id,
            "n_qubits":           n_qubits,
            "insertion_index":    insertion_index,
            "insertion_point":    "after_first_cp_rotation_block",
            "direction":          benign_meta.get("direction"),
            "do_swaps":           benign_meta.get("do_swaps"),
            "generation_index":   idx,
            "generation_seed":    seed,
            "generator_version":  GENERATOR_VERSION,
            "qiskit_version":     QISKIT_VERSION,
            "created":            datetime.now(timezone.utc).isoformat(),
            **trojan_meta,
        }

        with open(out_metadata_dir / f"{sample_id}.json", "w") as f:
            json.dump(metadata, f, indent=2)

        stats[trojan_type]["generated"] += 1

# ── Validation ────────────────────────────────────────────────

for trojan_type in TROJAN_TYPES:
    out_dir = (TAMPERED_DIR / FAMILY / DATASET_VERSION
               / trojan_type / "circuits")
    files   = sorted(out_dir.glob("*.qpy"))

    assert len(files) == NUM_BENIGN, \
        f"{trojan_type}: expected {NUM_BENIGN}, found {len(files)}"

    for f in files:
        with open(f, "rb") as fh:
            qc = qpy.load(fh)[0]
        assert qc.num_clbits == 0, \
            f"Measurement found in {f.name}"

print(f"QFT tampered generation complete.")
for trojan_type in TROJAN_TYPES:
    s = stats[trojan_type]
    print(f"  {trojan_type:10s} — generated: {s['generated']}, "
          f"skipped: {s['skipped']}")
print(f"  Total tampered : "
      f"{sum(s['generated'] for s in stats.values())} circuits")
print(f"  Validation     : all circuits clean ✓")


QFT tampered generation complete.
  static     — generated: 600, skipped: 0
  triggered  — generated: 600, skipped: 0
  Total tampered : 1200 circuits
  Validation     : all circuits clean ✓


In [9]:
# ============================================================
# CELL 8 — Full S2 Dataset Validation and Summary
# ============================================================
#
# Validates the complete S2 output across all five families
# and both Trojan types.
# Run this cell after all five generation cells complete.
#
# Checks:
#   - Correct circuit count per family per Trojan type
#   - No measurements in any tampered circuit
#   - Every tampered circuit has a matching metadata file
#   - Metadata schema is complete
#   - parent_sample_id links to a real benign circuit
#   - No duplicate sample_ids across all tampered circuits
#   - Qubit count matches benign parent (no ancilla added)
#   - Tampered circuit is larger than benign parent
#   - Saves s2_manifest.json
# ============================================================

REQUIRED_TAMPERED_FIELDS = [
    "sample_id",
    "label",
    "trojan_type",
    "algorithm_family",
    "algorithm_version",
    "parent_sample_id",
    "n_qubits",
    "insertion_index",
    "insertion_point",
    "mod_class",
    "payload_gate_count",
    "payload_description",
    "generation_index",
    "generation_seed",
    "generator_version",
    "qiskit_version",
    "created",
]

# ── Build benign circuit index for parent validation ─────────
# Maps sample_id → n_qubits and circuit size for all benign

print("Building benign circuit index...")
benign_index = {}

for family in FAMILIES:
    circuit_dir  = BENIGN_DIR / family / DATASET_VERSION / "circuits"
    metadata_dir = BENIGN_DIR / family / DATASET_VERSION / "metadata"

    for meta_file in sorted(metadata_dir.glob("*.json")):
        with open(meta_file) as f:
            meta = json.load(f)
        sid = meta["sample_id"]
        qpy_path = circuit_dir / f"{sid}.qpy"
        with open(qpy_path, "rb") as f:
            qc = qpy.load(f)[0]
        benign_index[sid] = {
            "n_qubits":    qc.num_qubits,
            "num_instructions": len(qc.data),
        }

print(f"  Benign index built: {len(benign_index)} circuits")

# ── Validation ────────────────────────────────────────────────

all_sample_ids = []
family_summary = {}
errors         = []

print("\n" + "=" * 55)
print("S2 — Full Dataset Validation")
print("=" * 55)

for family in FAMILIES:
    family_errors  = []
    family_stats   = {}

    for trojan_type in TROJAN_TYPES:
        circuit_dir  = (TAMPERED_DIR / family / DATASET_VERSION
                        / trojan_type / "circuits")
        metadata_dir = (TAMPERED_DIR / family / DATASET_VERSION
                        / trojan_type / "metadata")

        qpy_files  = sorted(circuit_dir.glob("*.qpy"))
        json_files = sorted(metadata_dir.glob("*.json"))

        type_errors = []

        # Count check
        if len(qpy_files) != NUM_BENIGN:
            type_errors.append(
                f"Circuit count: expected {NUM_BENIGN}, "
                f"found {len(qpy_files)}"
            )

        for qpy_file in qpy_files:
            sample_id = qpy_file.stem
            all_sample_ids.append(sample_id)

            # Load tampered circuit
            try:
                with open(qpy_file, "rb") as f:
                    qc_tampered = qpy.load(f)[0]
            except Exception as e:
                type_errors.append(
                    f"QPY load failed {sample_id}: {e}"
                )
                continue

            # No measurements
            if qc_tampered.num_clbits > 0:
                type_errors.append(
                    f"Measurement in {sample_id}"
                )

            # Load metadata
            meta_path = metadata_dir / f"{sample_id}.json"
            if not meta_path.exists():
                type_errors.append(
                    f"Missing metadata: {sample_id}"
                )
                continue

            with open(meta_path) as f:
                meta = json.load(f)

            # Schema check
            for field in REQUIRED_TAMPERED_FIELDS:
                if field not in meta:
                    type_errors.append(
                        f"{sample_id}: missing field '{field}'"
                    )

            # Label integrity
            if meta.get("label") != "tampered":
                type_errors.append(
                    f"{sample_id}: label should be 'tampered'"
                )

            if meta.get("trojan_type") != trojan_type:
                type_errors.append(
                    f"{sample_id}: trojan_type should be "
                    f"'{trojan_type}'"
                )

            # Parent linkage
            parent_id = meta.get("parent_sample_id")
            if not parent_id:
                type_errors.append(
                    f"{sample_id}: missing parent_sample_id"
                )
                continue

            if parent_id not in benign_index:
                type_errors.append(
                    f"{sample_id}: parent '{parent_id}' "
                    f"not in benign index"
                )
                continue

            parent_info = benign_index[parent_id]

            # Qubit count unchanged
            if qc_tampered.num_qubits != parent_info["n_qubits"]:
                type_errors.append(
                    f"{sample_id}: qubit count changed "
                    f"({parent_info['n_qubits']} → "
                    f"{qc_tampered.num_qubits})"
                )

            # Tampered circuit larger than benign
            if len(qc_tampered.data) <= parent_info["num_instructions"]:
                type_errors.append(
                    f"{sample_id}: tampered not larger than benign"
                )

        status = "✓ PASS" if not type_errors else "✗ FAIL"
        family_stats[trojan_type] = {
            "circuits": len(qpy_files),
            "status":   status,
            "errors":   type_errors,
        }
        family_errors.extend(type_errors)

    # Family summary
    overall = "✓ PASS" if not family_errors else "✗ FAIL"
    family_summary[family] = family_stats

    print(f"\n  {family} — {overall}")
    for trojan_type in TROJAN_TYPES:
        s = family_stats[trojan_type]
        print(f"    {trojan_type:10s} : "
              f"{s['circuits']} circuits — {s['status']}")
        if s["errors"]:
            for err in s["errors"][:3]:
                print(f"      ERROR: {err}")
            if len(s["errors"]) > 3:
                print(f"      ... and {len(s['errors']) - 3} more")

    errors.extend(family_errors)

# ── Duplicate sample_id check ─────────────────────────────────
seen       = set()
duplicates = []
for sid in all_sample_ids:
    if sid in seen:
        duplicates.append(sid)
    seen.add(sid)

if duplicates:
    errors.extend(
        [f"Duplicate sample_id: {d}" for d in duplicates]
    )

# ── Global summary ────────────────────────────────────────────
total_tampered = sum(
    family_summary[fam][tt]["circuits"]
    for fam in FAMILIES
    for tt in TROJAN_TYPES
)
total_expected = NUM_BENIGN * len(FAMILIES) * len(TROJAN_TYPES)

print("\n" + "=" * 55)
print(f"  Total tampered : {total_tampered} / {total_expected} expected")
print(f"  Duplicate IDs  : {len(duplicates)}")
print(f"  Total errors   : {len(errors)}")

if errors:
    print("\n  ✗ VALIDATION FAILED — fix errors before S3")
else:
    print("\n  ✓ ALL CHECKS PASSED — S2 complete")

# ── Save manifest ─────────────────────────────────────────────
manifest = {
    "notebook":          "S2_tampered_circuits.ipynb",
    "generator_version": GENERATOR_VERSION,
    "qiskit_version":    QISKIT_VERSION,
    "dataset_version":   DATASET_VERSION,
    "created":           datetime.now(timezone.utc).isoformat(),
    "total_tampered":    total_tampered,
    "total_expected":    total_expected,
    "families":          FAMILIES,
    "trojan_types":      TROJAN_TYPES,
    "validation_passed": len(errors) == 0,
    "per_family": {
        fam: {
            tt: {
                "circuits": family_summary[fam][tt]["circuits"],
                "status":   family_summary[fam][tt]["status"],
            }
            for tt in TROJAN_TYPES
        }
        for fam in FAMILIES
    },
}

manifest_path = TAMPERED_DIR / "s2_manifest.json"
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)

print(f"\n  Manifest saved : {manifest_path}")
print("=" * 55)


Building benign circuit index...
  Benign index built: 3000 circuits

S2 — Full Dataset Validation

  deutsch_jozsa — ✓ PASS
    static     : 600 circuits — ✓ PASS
    triggered  : 600 circuits — ✓ PASS

  grover — ✓ PASS
    static     : 600 circuits — ✓ PASS
    triggered  : 600 circuits — ✓ PASS

  qaoa — ✓ PASS
    static     : 600 circuits — ✓ PASS
    triggered  : 600 circuits — ✓ PASS

  vqc — ✓ PASS
    static     : 600 circuits — ✓ PASS
    triggered  : 600 circuits — ✓ PASS

  qft — ✓ PASS
    static     : 600 circuits — ✓ PASS
    triggered  : 600 circuits — ✓ PASS

  Total tampered : 6000 / 6000 expected
  Duplicate IDs  : 0
  Total errors   : 0

  ✓ ALL CHECKS PASSED — S2 complete

  Manifest saved : C:\Users\zajmal23\OneDrive - University of Oulu and Oamk\quantum_circuits_exp\dataset\circuits\tampered\s2_manifest.json
